# ChaturangaZero: 150 more iterations per session

Resumes the published 1,000-iteration run at learning rate 2e-4 (`configs/kaggle.yaml`).

**Before running:** Settings → Accelerator **GPU**, Internet **On**.
Then **Save Version → Save & Run All (Commit)** and close the tab.

Training stops on its own after 150 iterations **or 10.5 hours**, whichever comes first.
Every finished iteration is kept. Results end up in `/kaggle/working/final/`.

If the time limit stops training, a `BrokenPipeError` traceback from the worker
processes may appear in step 4. That is expected: the iteration in progress is
dropped, all finished ones are saved, and step 5 still collects the results.

In [ ]:
# Settings
NEW_ITERATIONS = 150              # this session trains this many more
TIME_LIMIT_HOURS = 10.5           # Kaggle kills sessions at 12 h; stop safely before that
REPO = "https://github.com/AditeyaAItronics/chaturangazero"
HF = "https://huggingface.co/ansh-k03/chaturangazero-500/resolve/main/run-1000"
RUN_DIR = "/kaggle/working/run"
LOG = "/kaggle/working/train.log"
INPUT_ROOT = "/kaggle/input"      # session 2: earlier output added via "Add Input"

In [ ]:
# 1. Check the GPU and get the code
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/chaturangazero
!git clone -q {REPO} /kaggle/working/chaturangazero
!cd /kaggle/working/chaturangazero && git log --oneline -1
!pip install -q pyyaml

In [ ]:
# 2. Starting point.
#    Session 2+: if an earlier session's output was added with "Add Input", continue from it.
#    Otherwise (session 1): download iteration 1,000, the iteration-870 incumbent and metrics.
import os, subprocess, glob, json, shutil
os.makedirs(RUN_DIR, exist_ok=True)
previous = [os.path.dirname(p) for p in glob.glob(INPUT_ROOT + "/**/final/run_state.json", recursive=True)]
if previous:
    src = max(previous, key=lambda d: json.load(open(os.path.join(d, "run_state.json")))
              ["last_completed_iteration"])
    state = json.load(open(os.path.join(src, "run_state.json")))
    print("Continuing from previous session:", src, "| last completed iteration",
          state["last_completed_iteration"] + 1)
    for name in os.listdir(src):
        if name.endswith((".pt", ".json", ".npz")):
            shutil.copy(os.path.join(src, name), RUN_DIR)
else:
    print("No previous session found: starting from the published iteration 1,000")
for name in ([] if previous else ["checkpoint_0999.pt", "best.pt", "metrics.json"]):
    dest = os.path.join(RUN_DIR, name)
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", f"{HF}/{name}", "-o", dest], check=True)
    print(name, os.path.getsize(dest) // 1024, "KB")

In [ ]:
# 3. Sanity check before spending 10 hours: the LR must come out at 2e-4
import sys
sys.path.insert(0, "/kaggle/working/chaturangazero")
os.chdir("/kaggle/working/chaturangazero")
from pipeline.loop import load_config
from pipeline.resume import latest_checkpoint
from net.model import load_net_from_checkpoint
from net.train import Trainer
import torch

cfg = load_config("configs/kaggle.yaml")
ckpt = latest_checkpoint(RUN_DIR)
net, _ = load_net_from_checkpoint(ckpt, "cpu")
t = Trainer(net, lr=cfg["train"]["lr"], lr_milestones=cfg["train"]["lr_milestones"], device="cpu")
t.load_checkpoint(ckpt)
t.reset_lr(cfg["train"]["lr"], cfg["train"]["lr_milestones"])
lr = t.optimizer.param_groups[0]["lr"]
start_iter = torch.load(ckpt, map_location="cpu", weights_only=False)["iteration"] + 1
TARGET_ITERATIONS = start_iter + NEW_ITERATIONS
print("Resuming from", os.path.basename(ckpt), f"(iteration {start_iter}) -> training to iteration {TARGET_ITERATIONS}")
print("LR after resume:", lr, "| CUDA available:", torch.cuda.is_available())
assert abs(lr - 2e-4) < 1e-12, "LR reset not working - do not train"
assert torch.cuda.is_available(), "No GPU - set Accelerator to GPU in Settings"
del net, t

In [ ]:
# 4. Train. Output goes to the cell and to train.log. Stops at TARGET_ITERATIONS or the time limit.
import time, signal, threading
cmd = [sys.executable, "-u", "train_chaturanga.py", "--config", "configs/kaggle.yaml",
       "--output", RUN_DIR, "--resume", "auto", "--iterations", str(TARGET_ITERATIONS)]
start = time.time()
deadline = start + TIME_LIMIT_HOURS * 3600
stopped_early = False
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def pump():   # copy training output to the cell and the log file
    with open(LOG, "a") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

reader = threading.Thread(target=pump, daemon=True)
reader.start()
while proc.poll() is None:
    if time.time() > deadline:
        print(f"\n*** Time limit of {TIME_LIMIT_HOURS} h reached: stopping training ***", flush=True)
        proc.send_signal(signal.SIGTERM)   # finished iterations are already saved
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        stopped_early = True
        break
    time.sleep(5)
reader.join(timeout=30)
print(f"\nTraining process ended (exit code {proc.returncode}) after "
      f"{(time.time() - start) / 3600:.2f} h{' - stopped by the time limit' if stopped_early else ''}")

In [ ]:
# 5. Collect the results into /kaggle/working/final (download this folder from the Output tab)
import json, shutil, glob
final = "/kaggle/working/final"
os.makedirs(final, exist_ok=True)
state_path = os.path.join(RUN_DIR, "run_state.json")
last = json.load(open(state_path))["last_completed_iteration"] if os.path.exists(state_path) else 999
shutil.copy(os.path.join(RUN_DIR, f"checkpoint_{last:04d}.pt"), final)
shutil.copy(os.path.join(RUN_DIR, "best.pt"), final)
shutil.copy(os.path.join(RUN_DIR, "metrics.json"), final)
shutil.copy(LOG, final)
for name in ["run_state.json", "buffer.npz"]:       # needed to continue in a later session
    if os.path.exists(os.path.join(RUN_DIR, name)):
        shutil.copy(os.path.join(RUN_DIR, name), final)

metrics = json.load(open(os.path.join(RUN_DIR, "metrics.json")))
new = [m for m in metrics if m["iteration"] >= start_iter]
best = torch.load(os.path.join(RUN_DIR, "best.pt"), map_location="cpu", weights_only=False)
print(f"Completed iterations: up to {last + 1} ({len(new)} new this session)")
print(f"best.pt is iteration {best['iteration'] + 1}")
print("Arena results this session:")
for m in new:
    a = m.get("arena")
    if a:
        print(f"  iter {m['iteration'] + 1}: {a['wins']}W {a['draws']}D {a['losses']}L "
              f"score {a['score']:.3f} p={a['p_value']}  {'PROMOTED' if a['promoted'] else ''}")
if new:
    print(f"Mean minutes per iteration: {sum(m['elapsed_seconds'] for m in new) / len(new) / 60:.1f}")
    print(f"Cap rate, last 10 iterations: {sum(m['cap_rate'] for m in new[-10:]) / len(new[-10:]):.0%}")
print("\nFiles to download:", sorted(os.listdir(final)))